# Automotive Data Mapper - MVP: Exploratory Data Analysis (EDA)

Date: 2026-08-06  
Author: Luis Renteria Lezano  
[LinkedIn](https://www.linkedin.com/in/renteria-luis) | [GitHub](https://github.com/renteria-luis) | [Portfolio](https://luisrenteria.me)

## Executive Summary

- Goal: explore the raw automotive service-record feeds before mapping them into one canonical schema, starting with the data types and formatting problems found in the Shop A feed. This notebook is the first step of the exploratory phase of a vehicle history record (VHR) style data-mapping project.
- **Sources:** Three synthetic vehicle service feeds representing an **independent repair shop**, a **dealership management system**, and a **fleet maintenance provider**:
  - `shop_a`: CSV
  - `dealer_b`: XML
  - `fleet_c`: JSON
- **Data:** [`../data/raw/`](https://github.com/renteria-luis/automotive-data-mapper/tree/main/data/raw)
- **Data dictionary:** [`../docs/DATA_DICTIONARY.md`](https://github.com/renteria-luis/automotive-data-mapper/blob/main/docs/DATA_DICTIONARY.md)
- **Sample data documentation:** [`../docs/SAMPLE_DATA.md`](https://github.com/renteria-luis/automotive-data-mapper/blob/main/docs/SAMPLE_DATA.md)

## 1. Reproducibility & Environment Setup

Import the library used for this exploration.

In [1]:
import pandas as pd

## 2. Load the Shop A Feed & First Look at Data Types

Read the raw CSV, check how pandas typed each column, and look at one full row up close.

In [2]:
df = pd.read_csv('../data/raw/shop_a/service_records_20260731.csv')
df.dtypes

VIN                       object
RO_OPEN_DATE              object
RO_CLOSE_DATE             object
MILEAGE                   object
ODOMETER_MEASURE          object
RO_INVOICE_NUMBER          int64
SERVICE_DESCRIPTION       object
LABOR_DESCRIPTION         object
PART_NAME_DESCRIPTION     object
PART_QUANTITY            float64
MAKE                      object
MODEL                     object
MODEL_YEAR               float64
PLATE                     object
PLATE_STATE               object
MANAGEMENT_SYSTEM         object
LOCATION_ID               object
LOCATION_NAME             object
ADDRESS                   object
CITY                      object
STATE                     object
POSTAL_CODE               object
PHONE                      int64
URL                       object
dtype: object

In [3]:
df['ODOMETER_MEASURE'].unique()

array(['KM', 'MI'], dtype=object)

In [4]:
df.iloc[0]

VIN                                        1FTFW1E50KFA12345
RO_OPEN_DATE                                      10/10/2019
RO_CLOSE_DATE                                     10/10/2019
MILEAGE                                               21,000
ODOMETER_MEASURE                                          KM
RO_INVOICE_NUMBER                                     184200
SERVICE_DESCRIPTION      Lube oil and filter, 5W30 synthetic
LABOR_DESCRIPTION        LUBE OIL AND FILTER, 5W30 SYNTHETIC
PART_NAME_DESCRIPTION                             OIL FILTER
PART_QUANTITY                                            1.0
MAKE                                                    FORD
MODEL                                                  F-150
MODEL_YEAR                                            2019.0
PLATE                                               CJKT 421
PLATE_STATE                                               ON
MANAGEMENT_SYSTEM                                 PROTRACTOR
LOCATION_ID             

Row 0 came through complete: `"21,000"` is stored as an `object`, not recognized as a number, and it stays whole in `MILEAGE` — it was not split at the comma, because `read_csv` respected the quotes.

> `object` means pandas is storing references to Python objects with no promise about each one's type. `MILEAGE` shows up as `object` simply because one cell, `"21,000"`, had a comma, and that was enough to push the whole column down to `object`. The `dtype` describes the column, not the values — knowing it is `object` does not say how many cells are `str`, how many are `float`, or anything else.

## 3. Why Whole-Number Columns Show Up as `float64`

`MODEL_YEAR` and `PART_QUANTITY` should hold whole numbers, but pandas reads them as `float64`. Take a closer look at a couple of rows to see why.

In [5]:
df.head(3)

,VIN,RO_OPEN_DATE,RO_CLOSE_DATE,MILEAGE,ODOMETER_MEASURE,RO_INVOICE_NUMBER,SERVICE_DESCRIPTION,LABOR_DESCRIPTION,PART_NAME_DESCRIPTION,PART_QUANTITY,...,PLATE_STATE,MANAGEMENT_SYSTEM,LOCATION_ID,LOCATION_NAME,ADDRESS,CITY,STATE,POSTAL_CODE,PHONE,URL
0,1FTFW1E50KFA12345,10/10/2019,10/10/2019,"21,000",KM,184200,"Lube oil and filter, 5W30 synthetic","LUBE OIL AND FILTER, 5W30 SYNTHETIC",OIL FILTER,1.0,...,ON,PROTRACTOR,CA-ON-4471,Riverside Auto Service,1247 Hamilton Rd,London,ON,N5W 1A7,5194553120,www.riversideautoservice.ca
1,1FTFW1E50KFA12345,06/03/2020,06/04/2020,30733,KM,184203,Replace front brake pads and machine rotors,REPLACE FRONT BRAKE PADS AND MACHINE ROT,BRAKE PAD SET,2.0,...,NaN,PROTRACTOR,CA-ON-4471,RIVERSIDE AUTO SERVICE,1247 Hamilton Rd,London,ON,N5W 1A7,5194553120,www.riversideautoservice.ca
2,2T1BURHE4JC021345,07/31/2019,08/02/2019,33255,KM,184206,"Rotate tires, adjust pressures","ROTATE TIRES, ADJUST PRESSURES",TIRE,4.0,...,ON,PROTRACTOR,CA-ON-4471,Riverside Auto Service Ltd,1247 Hamilton Rd,London,ON,N5W 1A7,5194553120,www.riversideautoservice.ca


On the other hand, `MODEL_YEAR` and `PART_QUANTITY` came out as `float64`, with values like `2019.0` — that does not mean the year is a decimal. `int64` cannot hold a `NaN`, so whenever a cell is empty, pandas converts the whole column to `float`. So here the `dtype` is a signal that there are nulls in the column:

In [6]:
df[(df['RO_INVOICE_NUMBER'] == 184290) | (df['RO_INVOICE_NUMBER'] == 184308)][['RO_INVOICE_NUMBER', 'MODEL_YEAR', 'PART_QUANTITY']]

,RO_INVOICE_NUMBER,MODEL_YEAR,PART_QUANTITY
30,184290,NaN,4.0
37,184308,NaN,2.0


In [7]:
df['PART_QUANTITY'].unique()

array([ 1.,  2.,  4.,  6.,  8., nan])

`read_csv` tries to fit the whole column into one numpy type: it tries `int64` first, then `float64`, and if neither works it gives up and leaves the column as `object`. The decision is made per column, and a single value is enough to push the whole column down.

## 4. Confirming What `object` Really Means for `MILEAGE`

Check the actual Python type behind individual `MILEAGE` values to see what `object` is hiding.

In [8]:
print(f"{df['MILEAGE'][0]} es {type(df['MILEAGE'][0])}")
print(f"{df['MILEAGE'][5]} es {type(df['MILEAGE'][5])}")

21,000 es <class 'str'>
71240 es <class 'str'>


Since pandas maps the whole column as `object`, every record becomes a `str`, not just `"21,000"` — the only rows that are not `str` are the `NaN` ones, and `NaN` is a `float`.

> So `object` does not mean the data is mixed — it means **pandas stopped making promises about this column**. That is why I have to count the types myself: a column-level guarantee says nothing about any individual value.

In [9]:
df['MILEAGE'].map(type).value_counts()

MILEAGE
<class 'str'>      41
<class 'float'>     1
Name: count, dtype: int64

This column is problematic — I will need to convert its 42 values to integers. I have to be careful with `.str.replace(',', '')` on the column: pandas will quietly return `NaN` for the `NaN` cell instead of raising an error, and a silent pass-through like that is exactly what I want to watch out for.

## 5. Cleaning the `MILEAGE` Column

Strip the thousands-separator commas and cast the column to a nullable integer type.

In [21]:
df['MILEAGE'] = df['MILEAGE'].str.replace(',', '')

In [31]:
df['MILEAGE'] = df['MILEAGE'].astype('Int64')
df['MILEAGE'].map(type).value_counts()

MILEAGE
<class 'float'>    42
Name: count, dtype: int64

In [32]:
df['MILEAGE'].isna().sum()

np.int64(1)